In [0]:
#@title Authenticate user
from google.colab import auth
auth.authenticate_user()

In [0]:
#@title Pull OpenAQ data via BigQuery
%%bigquery --project silicon-perigee-208521 p
#standardSQL
select
  pollutant,
  country,
  city,
  location,
  value
from `bigquery-public-data.openaq.global_air_quality`;

,pollutant,country,city,location,value
0,pm25,US,007,Plain_Mobile,0.000000
1,pm25,US,007,Cle Elum_Mobile,0.000000
2,pm25,US,007,Leavenworth_Mobile,10.000000
3,pm25,US,037,Mobile_Cle Elum,62.000000
4,pm25,US,039,Mobile_WhiteSalmon,58.000000
5,pm25,US,047,Brewster_Mobile,7.000000
6,pm25,US,051,Mobile_Newport,29.000000
7,pm25,US,051,Newport_Mobile,-2.000000
8,pm25,US,051,Mobile_Newport_2017,26.000000
9,co,US,ADA,Near Road,0.290000


In [0]:
#@title For each pollutant, calculate rank of each location within its country and world-wide 
# Rank within world
p = p.assign(world_rank=(
      p.groupby(['pollutant'])['value']
        .rank(method='min', ascending=False)
        .astype(int)))
# Rank within country
p = p.assign(country_rank=(
      p.groupby(['pollutant', 'country'])['value']
        .rank(method='min', ascending=False)
        .astype(int)))
# Rank within city
p = p.assign(city_rank=(
      p.groupby(['pollutant', 'country', 'city'])['value']
        .rank(method='min', ascending=False)
        .astype(int)))
p

,pollutant,country,city,location,value,world_rank,country_rank,city_rank
0,pm25,US,007,Plain_Mobile,0.000000,3757,964,2
1,pm25,US,007,Cle Elum_Mobile,0.000000,3757,964,2
2,pm25,US,007,Leavenworth_Mobile,10.000000,2222,253,1
3,pm25,US,037,Mobile_Cle Elum,62.000000,523,4,1
4,pm25,US,039,Mobile_WhiteSalmon,58.000000,556,5,1
5,pm25,US,047,Brewster_Mobile,7.000000,2631,435,1
6,pm25,US,051,Mobile_Newport,29.000000,1079,35,1
7,pm25,US,051,Newport_Mobile,-2.000000,3858,1009,3
8,pm25,US,051,Mobile_Newport_2017,26.000000,1191,40,2
9,co,US,ADA,Near Road,0.290000,2453,85,1


In [0]:
#@title Pollutants in dataset
print(', '.join(p['pollutant'].unique()))

pm25, co, bc, no2, pm10, o3, so2


In [0]:
#@title Top three locations by Nitrogen Dioxide (NO2)
(p.query('pollutant == "no2"')
  .nsmallest(3, 'world_rank')
  [['country', 'city', 'location', 'value', 'world_rank']])

,country,city,location,value,world_rank
19239,XK,Hani i Elezit,Hani i Elezit,397.926,1
6078,IN,Solapur,"Solapur, Solapur - MPCB",268.360,2
16204,IN,Kalaburagi,"Lal Bahadur Shastri Nagar, Kalaburagi - KSPCB",229.210,3


In [0]:
#@title Top three locations in Great Britain by Nitrogen Dioxide (NO2)
(p.query('pollutant == "no2" & country == "GB"')
  .nsmallest(3, 'country_rank')
  [['city', 'location', 'value', 'world_rank', 'country_rank']])

,city,location,value,world_rank,country_rank
4585,Belfast,Belfast Stockman's Lane,63.0,359,1
6707,Chepstow,Chepstow A48,60.0,418,2
6442,Aberdeen,Aberdeen Union Street Roadside,54.0,544,3


In [0]:
#@title Top three locations by Sulphur Dioxide (SO2)
(p.query('pollutant == "so2"')
  .nsmallest(3, 'world_rank')
  [['country', 'city', 'location', 'value', 'world_rank']])

,country,city,location,value,world_rank
24629,MK,State Air Quality Monitoring Network,Karpos,2.050019e+15,1
7092,CL,Hualpén,JUNJI,2.088350e+03,2
3412,ES,Murcia,ES1633A,1.301000e+03,3


In [0]:
#@title Top three locations in Great Britain by Sulphur Dioxide (SO2)
(p.query('pollutant == "so2" & country == "GB"')
  .nsmallest(3, 'country_rank')
  [['city', 'location', 'value', 'world_rank', 'country_rank']])

,city,location,value,world_rank,country_rank
19339,Middlesbrough,Middlesbrough,19.0,323,1
17408,Southampton,Southampton Centre,14.0,497,2
1188,Derry,Derry Rosemount,13.0,545,3


In [0]:
#@title Top three locations by PM2.5
(p.query('pollutant == "pm25"')
  .nsmallest(3, 'world_rank')
  [['country', 'city', 'location', 'value', 'world_rank']])

,country,city,location,value,world_rank
9849,CN,义乌市,江东,690.0,1
1695,IN,Patna,"IGSC Planetarium Complex, Patna - BSPCB",674.0,2
9845,CN,义乌市,北苑,494.0,3


In [0]:
#@title Top three locations in Great Britain by PM2.5
(p.query('pollutant == "pm25" & country == "GB"')
  .nsmallest(3, 'country_rank')
  [['city', 'location', 'value', 'world_rank', 'country_rank']])

,city,location,value,world_rank,country_rank
17442,Storrington,Storrington Roadside,62.0,523,1
19611,West Midlands,Birmingham A4540 Roadside,28.0,1118,2
8973,Newcastle,Newcastle Centre,17.0,1672,3


In [0]:
#@title Compare the worst locations (NO2) in Great Britain and Sweden
(p.query('pollutant == "no2"')
  .query('country_rank == 1')
  .query('(country == "GB") | (country == "SE")')
  [['country', 'city', 'location', 'value', 'world_rank', 'country_rank']])

,country,city,location,value,world_rank,country_rank
4585,GB,Belfast,Belfast Stockman's Lane,63.0000,359,1
9241,SE,Stockholm,Sveavägen,42.6175,917,1


Note that we can also do a lot of the work using BigQuery. But since the dataset is fairly small, we don't need to do that. Example on how to compute the rank in BigQuery:

In [0]:
#@title Rank cities within each country for each pollutant
%%bigquery --project silicon-perigee-208521
#standardSQL
select
  pollutant,
  country,
  city,
  location,
  value,
  rank() over (
    partition by
      pollutant,
      country
    order by value desc) as world_rank
from `bigquery-public-data.openaq.global_air_quality`;

,pollutant,country,city,location,value,world_rank
0,bc,PL,Kielce,"Kielce, ul. Targowa",5.73000,1
1,bc,PL,Rybnik,"Rybnik, ul. Borki 37 d",5.27091,2
2,bc,PL,Nowy Targ,"Nowy Targ, Plac Słowackiego",4.70168,3
3,bc,PL,Czerwionka-Leszczyny,"Czerwionka-Leszczyny, ul. Kopalniana",4.24980,4
4,bc,PL,Zamość,Zamość ul. Hrubieszowska 69A,2.98000,5
5,bc,PL,Zgorzelec,Zgorzelec - Bohaterów Getta,2.79247,6
6,bc,PL,Mielec,Mielec-Biernackiego-WIOS,2.44703,7
7,bc,PL,Łódź,Łódź-Gdańska 16,2.40000,8
8,bc,PL,Radom,Radom-Tochtermana,2.13700,9
9,bc,PL,Starachowice,"Starachowice, ul. Złota",2.05000,10
